# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
tf.__version__

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,shear_range = 0.2,zoom_range = 0.2,horizontal_flip = True)

training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

In [ ]:
print(prediction)

## Image Preprocessing and Data Augmentation

Before training a CNN, the images must be loaded in a consistent format and transformed into values the network can process effectively. In this notebook, preprocessing has two parts:

1. rescale both the training and evaluation images; and
2. apply random data augmentation only to the training images.

### Why augment the training set?

A model overfits when it learns the training examples too closely and fails to generalize to unseen data. One common warning sign is very high training accuracy accompanied by substantially lower validation or test accuracy.

Image augmentation reduces this risk by showing the network plausible variations of each training image. Instead of repeatedly seeing exactly the same pixels, the model sees randomly transformed versions and is encouraged to learn features that remain useful under small visual changes. This increases the effective diversity of the training data without requiring new labeled images.

The legacy generator in this notebook creates transformed images in memory as batches are requested; it does not normally save a permanently enlarged dataset to disk. Augmentation is helpful, but it does not guarantee that overfitting will disappear. Model capacity, regularization, data quality, and dataset size also matter.

### Transformations used in the notebook

```python
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
```

- `shear_range` applies a shear: an affine transformation that slants the image. A shear is not the same as a translation.
- `zoom_range` randomly zooms images in or out within the configured range.
- `horizontal_flip=True` randomly reflects images from left to right.
- `rescale=1.0 / 255` converts typical 8-bit pixel values from `[0, 255]` to `[0, 1]`.

Augmentations must preserve the meaning of the label. A horizontal flip is usually reasonable for cats and dogs, but it may be invalid when direction matters, such as reading text, recognizing left-versus-right road signs, or identifying laterality in medical images. Transformations should therefore be chosen for the problem rather than copied mechanically.

### Loading the training images

```python
training_set = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
)
```

`flow_from_directory` expects one subdirectory per class and infers labels from those directory names. Its main arguments here are:

- `target_size=(64, 64)`: resize every image to 64 by 64 pixels so that a batch has a consistent shape. Smaller images train faster but may discard useful detail.
- `batch_size=32`: provide 32 images at a time. This is a common starting point, not a universally optimal setting.
- `class_mode='binary'`: generate binary labels for a two-class task. This matches a model with one sigmoid output and binary cross-entropy loss.

Directory names are normally mapped to integer labels in alphanumeric order. The mapping should be checked rather than assumed:

```python
training_set.class_indices
```

### Preparing evaluation images

```python
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_set = test_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
)
```

Random augmentation is omitted from the evaluation pipeline because validation and test data should represent the unmodified examples on which the model must generalize. Deterministic preprocessing is still required: evaluation images must be resized and rescaled exactly as the training images are.

Dividing by the fixed constant 255 does not learn anything from the test set, so it does not itself cause data leakage. The familiar `fit_transform` versus `transform` distinction matters when preprocessing estimates quantities from training data, such as a mean and standard deviation. Here, the key requirement is simply to apply the same fixed input convention everywhere.

The notebook passes `test_set` as `validation_data` during training. Because its performance is observed after every epoch and may influence model choices, it is functioning as a **validation set**. For a strict final evaluation, keep a separate test set untouched until architecture and hyperparameter decisions are complete.

### Course API and modern Keras

This notebook uses the course-era `ImageDataGenerator` and `flow_from_directory` workflow. Current Keras code commonly loads directory data with `keras.utils.image_dataset_from_directory` and performs preprocessing with Keras layers. A modern equivalent is:

```python
import keras
from keras import layers

training_set = keras.utils.image_dataset_from_directory(
    'dataset/training_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
)

test_set = keras.utils.image_dataset_from_directory(
    'dataset/test_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
    shuffle=False,
)

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomZoom(0.2),
])

cnn = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Rescaling(1.0 / 255),
    data_augmentation,
    # Convolutional and classification layers follow.
])
```

When random augmentation layers are placed inside a Keras model, they apply during training and are inactive during inference. The deterministic `Rescaling` layer applies during both training and inference. The original notebook remains useful for learning the generator workflow, while this alternative reflects the current Keras data-loading style.


## Study Notes: Image Preprocessing

### Pipeline to remember

**Training:** load image -> resize -> rescale -> randomly augment -> batch -> CNN

**Validation/test:** load image -> resize -> rescale -> batch -> CNN

Only label-preserving random augmentation belongs in the training path. Deterministic transformations required by the model belong in every path, including single-image prediction.

### Key distinctions

| Concept | Meaning |
| --- | --- |
| Preprocessing | Deterministic preparation such as resizing or rescaling |
| Data augmentation | Random, label-preserving variation used during training |
| Batch | A group of examples processed before a parameter update |
| Epoch | One complete pass through the training dataset |
| Validation set | Data checked during model development |
| Test set | Data reserved for final, unbiased evaluation |
| Overfitting | Strong training performance but weak unseen-data performance |

### Parameter reference

| Setting | Purpose |
| --- | --- |
| `rescale=1./255` | Maps 8-bit image values from `[0, 255]` to `[0, 1]` |
| `target_size=(64, 64)` | Gives every image the spatial dimensions expected by the CNN |
| `batch_size=32` | Supplies 32 images per batch |
| `class_mode='binary'` | Produces labels for a legacy two-class generator |
| `label_mode='binary'` | Modern directory-dataset equivalent for binary labels |
| `horizontal_flip=True` | Adds random left-right reflections |
| `zoom_range=0.2` | Adds random zoom variation |
| `shear_range=0.2` | Adds random affine shearing in the legacy generator |

### Practical checks

- Confirm that each class has its own correctly named subdirectory.
- Inspect several augmented images to ensure that labels still make sense.
- Verify the inferred class-to-index mapping.
- Use the same image size and deterministic scaling during training, evaluation, and prediction.
- Do not choose augmentation ranges so aggressive that objects become unrealistic or class information disappears.
- Monitor both training and validation curves rather than relying on training accuracy alone.
- Keep a separate final test set when an unbiased performance estimate is required.
- Set evaluation-data shuffling to `False` when predictions must align with filenames or labels in a fixed order.

### Quick self-check

1. Why is random augmentation applied only to training data?
2. Why must evaluation images still be resized and rescaled?
3. Does `ImageDataGenerator` necessarily create new files on disk?
4. What model output and loss match binary labels?
5. Why might horizontal flipping be harmful for some datasets?
6. What is the trade-off when reducing images from 150 by 150 to 64 by 64?
7. Why is a dataset used as `validation_data` no longer a pristine final test set?

### Answers

1. Augmentation regularizes learning by exposing the model to varied training examples; evaluation should measure performance on the natural, unmodified data distribution.
2. The model requires the same shape and numerical input convention it saw during training.
3. No. In this workflow, augmented variants are normally generated in memory as batches are requested.
4. A single sigmoid output with binary cross-entropy.
5. A flip may change the label or remove task-relevant directional meaning.
6. Smaller images reduce computation and training time but can remove fine visual detail.
7. Repeatedly observing its results can influence model decisions, indirectly fitting the development process to that dataset.

### Final takeaway

Resizing and rescaling make inputs consistent; augmentation makes the training data more varied. A sound pipeline augments only training images, applies identical deterministic preprocessing everywhere, and preserves truly unseen data for final evaluation.
